# Top-1 S&P 500 Weight Monthly Strategy

**Rule:** At the start of each month, read the latest `sp500_constituents_YYYYMM.csv`, pick the stock with `rank == 1` (highest index weight), and invest 100 % of the portfolio in it for the entire month.

Historical prices are fetched from Yahoo Finance via `yfinance`.

**Benchmark:** SPY (S&P 500 ETF)

> The strategy rebalances on the **first trading day** of each month and holds through the last trading day of that month.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import pandas as pd
import yfinance as yf

CONSTITUENTS_DIR = Path("sp500_constituents")
INITIAL_CAPITAL   = 10_000   # USD
BENCHMARK         = "SPY"

## 1 · Build the monthly holdings table

One row per CSV file → the top-1 stock for that month.

In [ ]:
records = []
for csv_file in sorted(CONSTITUENTS_DIR.glob("sp500_constituents_*.csv")):
    df = pd.read_csv(csv_file)
    top1 = df.loc[df["rank"] == 1, ["symbol", "company", "weight_pct", "date"]].iloc[0]
    # derive the month from the filename (YYYYMM)
    month_tag = csv_file.stem.split("_")[-1]          # e.g. '202603'
    records.append({
        "month"     : pd.to_datetime(month_tag, format="%Y%m"),
        "symbol"    : top1["symbol"],
        "company"   : top1["company"],
        "weight_pct": top1["weight_pct"],
    })

holdings = pd.DataFrame(records).sort_values("month").reset_index(drop=True)
print(f"Months loaded: {len(holdings)}")
display(holdings)

## 2 · Download price data

In [ ]:
all_symbols = sorted(set(holdings["symbol"].tolist() + [BENCHMARK]))

start_date = holdings["month"].min().strftime("%Y-%m-01")
# fetch one extra month so the last holding has a full period
end_date   = (holdings["month"].max() + pd.DateOffset(months=2)).strftime("%Y-%m-01")

print(f"Downloading {all_symbols} from {start_date} to {end_date} …")
raw = yf.download(all_symbols, start=start_date, end=end_date,
                  auto_adjust=True, progress=False)

# keep only closing prices; flatten multi-index if multiple tickers
if isinstance(raw.columns, pd.MultiIndex):
    prices = raw["Close"].copy()
else:
    prices = raw[["Close"]].rename(columns={"Close": all_symbols[0]}).copy()

prices.index = pd.to_datetime(prices.index)
prices.sort_index(inplace=True)
print(f"Price matrix shape: {prices.shape}")
prices.tail(3)

## 3 · Backtest

In [ ]:
def monthly_return(prices_series: pd.Series, entry_date: pd.Timestamp, exit_date: pd.Timestamp) -> float:
    """Return the simple return between the first available price on/after
    entry_date and the last available price on/before exit_date."""
    window = prices_series.loc[entry_date:exit_date].dropna()
    if len(window) < 2:
        return 0.0
    return window.iloc[-1] / window.iloc[0] - 1


results = []
strat_value = INITIAL_CAPITAL
bench_value = INITIAL_CAPITAL

for i, row in holdings.iterrows():
    entry = row["month"]                                          # first day of that month
    if i + 1 < len(holdings):
        exit_ = holdings.loc[i + 1, "month"] - pd.Timedelta(days=1)  # last day before next month
    else:
        exit_ = entry + pd.offsets.MonthEnd(1)                   # end of this month

    sym = row["symbol"]
    if sym not in prices.columns:
        print(f"  [WARN] {sym} not in price data — skipping month {entry:%Y-%m}")
        continue

    strat_ret = monthly_return(prices[sym],       entry, exit_)
    bench_ret = monthly_return(prices[BENCHMARK], entry, exit_)

    strat_value *= (1 + strat_ret)
    bench_value *= (1 + bench_ret)

    results.append({
        "month"       : entry,
        "symbol"      : sym,
        "company"     : row["company"],
        "weight_pct"  : row["weight_pct"],
        "strat_return": round(strat_ret * 100, 2),
        "bench_return": round(bench_ret * 100, 2),
        "strat_value" : round(strat_value, 2),
        "bench_value" : round(bench_value, 2),
    })

perf = pd.DataFrame(results)
display(perf)

## 4 · Summary statistics

In [ ]:
if not perf.empty:
    total_strat = (perf["strat_value"].iloc[-1] / INITIAL_CAPITAL - 1) * 100
    total_bench = (perf["bench_value"].iloc[-1] / INITIAL_CAPITAL - 1) * 100
    avg_strat   = perf["strat_return"].mean()
    avg_bench   = perf["bench_return"].mean()

    print(f"{'Metric':<30} {'Strategy':>12} {'SPY Benchmark':>15}")
    print("-" * 60)
    print(f"{'Initial capital':<30} {'$'+f'{INITIAL_CAPITAL:,.0f}':>12} {'$'+f'{INITIAL_CAPITAL:,.0f}':>15}")
    print(f"{'Final value':<30} {'$'+f'{perf.strat_value.iloc[-1]:,.2f}':>12} {'$'+f'{perf.bench_value.iloc[-1]:,.2f}':>15}")
    print(f"{'Total return (%)':<30} {total_strat:>11.2f}% {total_bench:>14.2f}%")
    print(f"{'Avg monthly return (%)':<30} {avg_strat:>11.2f}% {avg_bench:>14.2f}%")
    print(f"{'Months tracked':<30} {len(perf):>12}")

## 5 · Portfolio value chart

In [ ]:
if not perf.empty:
    fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True,
                             gridspec_kw={"height_ratios": [3, 1]})

    # ── top panel: portfolio value ──────────────────────────────────────
    ax = axes[0]
    # prepend the starting point
    dates  = [perf["month"].iloc[0] - pd.DateOffset(months=1)] + perf["month"].tolist()
    s_vals = [INITIAL_CAPITAL] + perf["strat_value"].tolist()
    b_vals = [INITIAL_CAPITAL] + perf["bench_value"].tolist()

    ax.plot(dates, s_vals, marker="o", linewidth=2, label="Top-1 Strategy", color="steelblue")
    ax.plot(dates, b_vals, marker="s", linewidth=2, label="SPY Benchmark",  color="darkorange",
            linestyle="--")
    ax.axhline(INITIAL_CAPITAL, color="grey", linewidth=0.8, linestyle=":")
    ax.yaxis.set_major_formatter(mticker.StrMethodFormatter("${x:,.0f}"))
    ax.set_ylabel("Portfolio Value (USD)")
    ax.set_title("Top-1 S&P 500 Weight Strategy  vs  SPY", fontsize=13, fontweight="bold")
    ax.legend()
    ax.grid(axis="y", alpha=0.4)

    # annotate held ticker each month
    for _, r in perf.iterrows():
        ax.annotate(r["symbol"], xy=(r["month"], r["strat_value"]),
                    xytext=(0, 8), textcoords="offset points",
                    ha="center", fontsize=7.5, color="steelblue")

    # ── bottom panel: monthly returns ───────────────────────────────────
    ax2 = axes[1]
    width = 10  # days
    ax2.bar([d - pd.Timedelta(days=6) for d in perf["month"]], perf["strat_return"],
            width=width, label="Strategy", color="steelblue", alpha=0.8)
    ax2.bar([d + pd.Timedelta(days=6) for d in perf["month"]], perf["bench_return"],
            width=width, label="SPY",      color="darkorange", alpha=0.8)
    ax2.axhline(0, color="black", linewidth=0.7)
    ax2.yaxis.set_major_formatter(mticker.StrMethodFormatter("{x:.1f}%"))
    ax2.set_ylabel("Monthly Return")
    ax2.legend(fontsize=8)
    ax2.grid(axis="y", alpha=0.4)

    plt.tight_layout()
    plt.savefig("sp500_constituents/top1_strategy.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("Chart saved → sp500_constituents/top1_strategy.png")